# Notebook ML — Pipeline de entrenamiento Predikt

Visualización del flujo completo de machine learning:

1. Validación de datos en `raw/`
2. Construcción de features y etiquetas (up/down a 1 día)
3. Partición temporal train / test
4. Entrenamiento y comparación de modelos (sklearn)
5. Métricas, matrices de confusión e importancia de features

Ejecutar con el kernel **Python 3 (predikt)** desde la raíz del repo o desde `notebooks/`.

## 1. Setup y rutas

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dataset import (
    DEFAULT_MIN_PRICES,
    DEFAULT_TRAIN_RATIO,
    FEATURE_COLS,
    build_training_matrix,
    eligible_slugs,
    split_panel_temporal,
    validate_markets,
)
from src.models import build_all_models, model_names
from src.paths import MODELS_DIR, PROCESSED_DIR, RAW_DIR
from src.training import evaluate_classifier, find_best_threshold, majority_baseline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

print(f"ROOT:      {ROOT}")
print(f"RAW_DIR:   {RAW_DIR}  (exists={RAW_DIR.is_dir()})")
print(f"PROCESSED: {PROCESSED_DIR}")
print(f"MODELS:    {MODELS_DIR}")

## 2. Paso 1 — Validación del dataset (`raw/`)

Equivalente a `python scripts/01_validate_dataset.py`.

In [ ]:
MIN_PRICES = DEFAULT_MIN_PRICES  # 30
TRAIN_RATIO = DEFAULT_TRAIN_RATIO  # 0.70

report = validate_markets(RAW_DIR, min_prices=MIN_PRICES)
slugs = eligible_slugs(report)

n_total = len(report)
n_eligible = len(slugs)
print(f"Archivos prices_*.csv: {n_total}")
print(f"Elegibles (n_prices >= {MIN_PRICES}): {n_eligible}")
print(f"Excluidos: {n_total - n_eligible}")

report.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

eligible_mask = report["eligible"].astype(bool)
axes[0].hist(report.loc[eligible_mask, "n_prices"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Distribución n_prices (elegibles)")
axes[0].set_xlabel("Días de precio")

counts = report["eligible"].value_counts()
axes[1].bar(["Elegible", "Excluido"], [counts.get(True, 0), counts.get(False, 0)], color=["#2ecc71", "#e74c3c"])
axes[1].set_title("Mercados elegibles vs excluidos")

if "is_closed" in report.columns:
    sub = report.loc[eligible_mask]
    closed = sub["is_closed"].fillna(False).astype(bool).value_counts()
    axes[2].bar(closed.index.astype(str), closed.values, color=["#3498db", "#f39c12"])
    axes[2].set_title("Elegibles: is_closed (catálogo)")
else:
    axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. Paso 2 — Matriz de entrenamiento (features + label)

- **Features:** retornos, medias móviles, volatilidad, precio YES.
- **Label:** `1` = precio sube al día siguiente, `0` = baja o igual.

In [ ]:
matrix_path = PROCESSED_DIR / "train_matrix.csv"
if matrix_path.is_file():
    matrix = pd.read_csv(matrix_path, parse_dates=["date"])
    print(f"Cargado desde {matrix_path.name}: {len(matrix):,} filas")
else:
    matrix = build_training_matrix(slugs, raw_dir=RAW_DIR)
    print(f"Construido en memoria: {len(matrix):,} filas")

matrix["date"] = pd.to_datetime(matrix["date"], utc=True)
print(f"Mercados: {matrix['slug'].nunique()}")
print(f"Rango fechas: {matrix['date'].min().date()} → {matrix['date'].max().date()}")
matrix.head()

In [ ]:
label_counts = matrix["label"].value_counts().sort_index()
label_pct = matrix["label"].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Down (0)", "Up (1)"], label_counts.values, color=["#e74c3c", "#27ae60"])
for b, pct in zip(bars, label_pct.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{pct:.1%}", ha="center", va="bottom")
ax.set_title("Distribución global de etiquetas (panel completo)")
ax.set_ylabel("Filas (día × mercado)")
plt.show()

display(label_counts.to_frame("count").assign(pct=label_pct))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
corr = matrix[FEATURE_COLS + ["label"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlación entre features y label")
plt.tight_layout()
plt.show()

## 4. Partición temporal (70% train / 30% test por mercado)

Sin shuffle: en cada mercado, los **últimos** días van a test. Simula predecir el futuro con datos solo del pasado.

In [ ]:
train_df, test_df = split_panel_temporal(matrix, train_ratio=TRAIN_RATIO)

print(f"Train: {len(train_df):,} filas | Test: {len(test_df):,} filas")
print(f"Train label up%: {train_df['label'].mean():.2%}")
print(f"Test label up%:  {test_df['label'].mean():.2%}")

X_train = train_df[FEATURE_COLS].values
y_train = train_df["label"].values.astype(int)
X_test = test_df[FEATURE_COLS].values
y_test = test_df["label"].values.astype(int)

In [ ]:
example_slug = matrix["slug"].value_counts().index[0]
g = matrix[matrix["slug"] == example_slug].sort_values("date")
split_idx = int(len(g) * TRAIN_RATIO)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(g["date"], g["price"], color="gray", alpha=0.4, label="precio")
ax.axvline(g.iloc[split_idx]["date"], color="black", ls="--", lw=1.5, label="corte 70/30")
ax.scatter(g.iloc[:split_idx]["date"], g.iloc[:split_idx]["price"], s=8, c="#3498db", label="train")
ax.scatter(g.iloc[split_idx:]["date"], g.iloc[split_idx:]["price"], s=8, c="#e67e22", label="test")
ax.set_title(f"Split temporal — ejemplo: {example_slug[:50]}…")
ax.legend(loc="upper left")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 5. Métricas guardadas (`processed/models/`)

Cargadas desde los scripts `02_train_baseline.py` y `03_compare_models.py`.

In [ ]:
metrics_path = MODELS_DIR / "metrics.json"
comparison_path = MODELS_DIR / "model_comparison.json"

metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.is_file() else {}
comparison = json.loads(comparison_path.read_text(encoding="utf-8")) if comparison_path.is_file() else {}

if metrics:
    print("=== metrics.json ===")
    print(f"Modelo guardado: {metrics.get('model')}")
    print(f"Umbral: {metrics.get('threshold', 0.5)}")
    print(f"Train rows: {metrics.get('n_train_rows')} | Test rows: {metrics.get('n_test_rows')}")
else:
    print("No hay metrics.json — ejecuta scripts/02 o 03 primero.")

if comparison:
    print(f"\nComparación disponible: {list(comparison.get('models', {}).keys())}")

In [ ]:
def metrics_to_table(source: dict, prefix: str = "") -> pd.DataFrame:
    rows = []
    if "all_models_test" in source:
        for name, m in source["all_models_test"].items():
            rows.append({"model": name, **m})
    elif "models" in source:
        for name, m in source["models"].items():
            rows.append({
                "model": name,
                "accuracy": m.get("accuracy"),
                "balanced_accuracy": m.get("balanced_accuracy"),
                "f1_macro": m.get("f1_macro"),
                "threshold": m.get("threshold", 0.5),
            })
    if "majority_baseline_test" in source:
        m = source["majority_baseline_test"]
        rows.insert(0, {
            "model": "majority (always down)",
            "accuracy": m["accuracy"],
            "balanced_accuracy": m.get("balanced_accuracy", 0.5),
            "f1_macro": m["f1_macro"],
            "threshold": "—",
        })
    return pd.DataFrame(rows)

src = comparison if comparison else metrics
metrics_table = metrics_to_table(src)
metrics_table.style.background_gradient(subset=["accuracy", "balanced_accuracy", "f1_macro"], cmap="YlGn")

In [ ]:
if not metrics_table.empty:
    plot_df = metrics_table[metrics_table["model"] != "majority (always down)"].copy()
    x = np.arange(len(plot_df))
    w = 0.25

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w, plot_df["accuracy"], width=w, label="accuracy")
    ax.bar(x, plot_df["balanced_accuracy"], width=w, label="balanced_accuracy")
    ax.bar(x + w, plot_df["f1_macro"], width=w, label="f1_macro")

    if "majority_baseline_test" in (comparison or metrics):
        maj_acc = (comparison or metrics)["majority_baseline_test"]["accuracy"]
        ax.axhline(maj_acc, color="red", ls="--", lw=1, label=f"majority acc={maj_acc:.3f}")

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model"], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title("Comparación de modelos en conjunto de test (temporal)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Entrenamiento en notebook (sklearn)

Replica `03_compare_models.py` con visualización. Desactiva `RUN_TRAINING = False` para solo ver artefactos guardados.

In [ ]:
RUN_TRAINING = True  # False → solo cargar best_model.pkl
TUNE_THRESHOLD = True

val_size = max(int(len(X_train) * 0.15), 500)
X_fit, y_fit = X_train[:-val_size], y_train[:-val_size]
X_val, y_val = X_train[-val_size:], y_train[-val_size:]

results = {}
fitted_models = {}

if RUN_TRAINING:
    for name, est in build_all_models(y_fit).items():
        print(f"Entrenando {name}…")
        est.fit(X_fit, y_fit)
        threshold = 0.5
        if TUNE_THRESHOLD and hasattr(est, "predict_proba"):
            threshold, _ = find_best_threshold(est, X_val, y_val, metric="accuracy")
        test_m = evaluate_classifier(est, X_test, y_test, threshold=threshold)
        test_m["threshold"] = threshold
        results[name] = test_m
        fitted_models[name] = est
    results["majority_baseline"] = majority_baseline(y_test)
else:
    pkl_path = MODELS_DIR / "best_model.pkl"
    if pkl_path.is_file():
        bundle = joblib.load(pkl_path)
        print(f"Cargado: {bundle.get('model_name', 'unknown')}")
    else:
        print("No hay best_model.pkl")

In [ ]:
if results:
    cm_fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    axes = axes.ravel()
    model_list = [k for k in results if k != "majority_baseline"][:4]

    for ax, name in zip(axes, model_list):
        cm = np.array(results[name]["confusion_matrix"])
        disp = ConfusionMatrixDisplay(cm, display_labels=["down", "up"])
        disp.plot(ax=ax, colorbar=False, cmap="Blues")
        acc = results[name]["accuracy"]
        ax.set_title(f"{name}\naccuracy={acc:.3f}")

    plt.suptitle("Matrices de confusión — test (30% final por mercado)", y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Modelo guardado — predicciones y importancia (árboles)

Carga `best_model.pkl` y analiza errores en test.

In [ ]:
bundle_path = MODELS_DIR / "best_model.pkl"
if bundle_path.is_file():
    bundle = joblib.load(bundle_path)
    model = bundle["model"]
    threshold = bundle.get("threshold", 0.5)
    model_name = bundle.get("model_name", "unknown")
    print(f"Modelo: {model_name} | threshold={threshold}")

    test_metrics = evaluate_classifier(model, X_test, y_test, threshold=threshold)
    print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Balanced acc:  {test_metrics['balanced_accuracy']:.4f}")
    print(test_metrics["classification_report"])
else:
    print("Ejecuta scripts/03_compare_models.py o activa RUN_TRAINING.")

In [ ]:
clf = None
if bundle_path.is_file():
    m = bundle["model"]
    if hasattr(m, "feature_importances_"):
        clf = m
    elif hasattr(m, "named_steps") and hasattr(m.named_steps.get("clf", m), "feature_importances_"):
        clf = m.named_steps["clf"]

if clf is not None:
    imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Importancia de features — {model_name}")
    ax.set_xlabel("importance")
    plt.tight_layout()
    plt.show()
else:
    print("El modelo guardado no expone feature_importances_ (p. ej. solo logística en pipeline).")

In [ ]:
if bundle_path.is_file() and hasattr(model, "predict_proba"):
    proba_up = model.predict_proba(X_test)[:, 1]
    pred = (proba_up >= threshold).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(proba_up[y_test == 0], bins=40, alpha=0.7, label="real down", color="#e74c3c")
    axes[0].hist(proba_up[y_test == 1], bins=40, alpha=0.7, label="real up", color="#27ae60")
    axes[0].axvline(threshold, color="black", ls="--", label=f"umbral={threshold}")
    axes[0].set_title("Distribución P(up) en test")
    axes[0].legend()

    err_mask = pred != y_test
    axes[1].scatter(test_df.loc[~err_mask, "ret_1d"], proba_up[~err_mask], s=5, alpha=0.3, label="acierto")
    axes[1].scatter(test_df.loc[err_mask, "ret_1d"], proba_up[err_mask], s=8, alpha=0.6, c="red", label="error")
    axes[1].set_xlabel("ret_1d")
    axes[1].set_ylabel("P(up)")
    axes[1].set_title("P(up) vs ret_1d")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 8. Resumen del pipeline (statsmodels — regresión logística)

Modelo lineal interpretable sobre features escaladas (solo train, sin leakage de test).

In [ ]:
try:
    import statsmodels.api as sm
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_te_s = scaler.transform(X_test)

    X_tr_sm = sm.add_constant(X_tr_s)
    logit = sm.Logit(y_train, X_tr_sm)
    logit_res = logit.fit(disp=False, maxiter=200)

    coef_names = ["const"] + FEATURE_COLS
    coef_df = pd.DataFrame({
        "coef": logit_res.params,
        "pvalue": logit_res.pvalues,
        "odds_ratio": np.exp(logit_res.params),
    }, index=coef_names)
    display(coef_df.round(4))

    proba_sm = logit_res.predict(sm.add_constant(X_te_s))
    pred_sm = (proba_sm >= 0.5).astype(int)
    print(f"\nStatsmodels Logit — test accuracy: {(pred_sm == y_test).mean():.4f}")
    print(logit_res.summary().tables[0])
except ImportError:
    print("statsmodels no instalado. pip install statsmodels")

## 9. Checklist y comandos CLI

| Paso | Script | Salida |
|------|--------|--------|
| Validar raw | `python scripts/01_validate_dataset.py` | `processed/dataset_validation_report.csv` |
| Baseline LR | `python scripts/02_train_baseline.py` | `processed/models/baseline_v1.pkl` |
| Comparar modelos | `python scripts/03_compare_models.py --tune-threshold` | `model_comparison.json`, `best_model.pkl` |

**Interpretación:** accuracy alta con clase desbalanceada puede igualar "siempre down". Revisar `balanced_accuracy` y matriz de confusión antes de usar en Predikt Now.